# Analiza dialogowa scenariusza: z jakim ładunkiem emocjonalnym?

## Wersja studencka — moduł 3: EMOCJE (wariant B — OpenRouter / model darmowy)

W poprzednim module wydobyłeś kwestie dialogowe i zmierzyłeś, *kto* mówi i *ile*. Teraz pytamy o *jak* — z jakim ładunkiem emocjonalnym. Do oceny emocji użyjemy darmowego modelu językowego dostępnego przez **OpenRouter** — serwis agregujący dziesiątki modeli pod jednym API kompatybilnym z OpenAI.

Efektem końcowym będą:
1. wczytanie kwestii z `dialogi.csv` i danych o postaciach z `postacie.csv`,
2. konfiguracja połączenia z OpenRouter API (darmowy klucz),
3. batchowa ocena emocji wszystkich kwestii,
4. łuk emocjonalny filmu (walencja wzdłuż scen),
5. wykres rozkładu emocji według płci postaci,
6. zapis pliku `emocje.csv` do dalszej analizy.

## Jak pracować z tym notatnikiem

1. Upewnij się, że masz w tym samym środowisku pliki `dialogi.csv` i `postacie.csv` z NB1.
2. Skopiuj prompt z bieżącego kroku do modelu AI — **ale najpierw przeczytaj go i uzupełnij luki**.
3. Wklej otrzymany kod do pustej komórki pod promptem.
4. Uruchom kod i porównaj z sekcją **Po uruchomieniu powinieneś zobaczyć**.
5. Dopiero po uzyskaniu poprawnego wyniku przejdź dalej.

**Uwaga do kroku z API:** w Etapie 2 będziesz potrzebować bezpłatnego klucza OpenRouter — zarejestruj się na openrouter.ai (konto darmowe, klucz gotowy w kilka minut). Klucz jest prywatny — nie wklejaj go na stałe do komórek.

## Twoja kolej: uzupełniaj prompty

W NB1 czytałeś gotowe prompty i uczyłeś się rozumieć ich 7-sekcyjną strukturę. Teraz kilka promptów ma **luki oznaczone `[UZUPEŁNIJ: …]`**, które musisz wypełnić.

**Jak to działa:**
Zamiast tekstu w markdown, prompty żyją teraz w zmiennej `moj_prompt` w komórce kodu.
1. Przeczytaj sekcję **Cel i sens analityczny** — opisuje, *co* chcemy osiągnąć.
2. Przeczytaj sekcję **Po uruchomieniu powinieneś zobaczyć** — mówi, *jak* powinien wyglądać wynik.
3. **Edytuj zmienną `moj_prompt`** — wypełnij każdą lukę `[UZUPEŁNIJ: …]`.
4. **Uruchom komórkę** — asercja sprawdzi, czy wszystkie luki są wypełnione i prompt ma odpowiednią treść.
5. Jeśli asercja przejdzie, skopiuj wypisany prompt do asystenta AI.

Zasada: luka `[UZUPEŁNIJ: …]` opisuje **co** powinieneś tam wpisać — ale Ty decydujesz, **jak** to sformulujesz.

---
## Etap 1/6 — Wczytanie danych wejściowych

Zaczynamy od danych z NB1.

In [ ]:
# === ETAP 1/6 · WCZYTANIE DANYCH ===
import os
brak = [f for f in ["dialogi.csv", "postacie.csv"] if not os.path.exists(f)]
if brak:
    raise FileNotFoundError(
        f"⛔ Brakuje plików: {brak}\n"
        "Pobierz je z NB1 (Etap 6) i wgraj do tego środowiska."
    )
print("📍 Etap 1/6 · Wczytanie danych. Pliki dialogi.csv i postacie.csv — znalezione ✓")

### Krok 1A. Wczytanie `dialogi.csv` i `postacie.csv`

#### Cel i sens analityczny

`dialogi.csv` zawiera po jednym wierszu na kwestię. `postacie.csv` zawiera podsumowanie postaci z płcią. Musimy załadować oba i sprawdzić, że wyglądają poprawnie.

#### Prompt dla modelu

```text
Kontekst:
Masz dwa pliki CSV z poprzedniego modułu analizy dialogowej.

Wejście:
Pliki `dialogi.csv` i `postacie.csv` w kodowaniu UTF-8.

Zadanie:
Wczytaj oba pliki do tabel i wyświetl krótki podgląd każdego z nich. Zachowaj wczytane tabele jako zmienne o nazwach `dialogi` i `postacie`.

Pokaż wynik:
- liczbę wierszy i nazwy kolumn każdego pliku,
- pierwsze 5 wierszy z `dialogi`,
- wszystkie wiersze z `postacie` (jest ich mało).

Warunek poprawności:
Kolumny powinny odpowiadać opisanemu formatowi. Jeśli nazwy kolumn są inne, wypisz to wprost.

Jeśli wystąpi błąd:
Wyjaśnij, którego pliku nie udało się wczytać i z jakiego powodu.

Nie rób jeszcze:
Nie oceniaj emocji.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Liczbę wierszy i kolumny obu plików.
- Kilka pierwszych kwestii z `dialogi`.
- Tabelę postaci z kolumną płeć (K/M/?).

---
## Etap 2/6 — Konfiguracja modelu językowego

Instalujemy bibliotekę `openai` (działa z OpenRouter bez żadnych zmian), podłączamy bezpłatny klucz API i sprawdzamy połączenie.

In [ ]:
# === ETAP 2/6 · KONFIGURACJA OPENROUTER ===
assert 'dialogi' in dir(), "⛔ Najpierw uruchom Etap 1 — brak tabeli `dialogi`."
print(f"📍 Etap 2/6 · Konfiguracja OpenRouter. Wejście: `dialogi` ({len(dialogi):,} kwestii).")

### Krok 2A. Instalacja i klucz API

#### Cel i sens analityczny

OpenRouter udostępnia wiele modeli (w tym darmowe) przez API kompatybilne z OpenAI — używamy tej samej biblioteki `openai`, ale z adresem `https://openrouter.ai/api/v1`. Klucz generujesz bezpłatnie na openrouter.ai i wpisujesz przez `getpass`, żeby nie pojawił się w historii komórek.

#### Jak wybrać model

1. Wejdź na **https://openrouter.ai/models**
2. Posortuj od **Most Popular** (najpopularniejsze modele zazwyczaj lepiej radzą sobie ze strukturyzowanym JSON-em)
3. Ustaw maksymalną cenę na **0** — zostaną tylko modele całkowicie darmowe
4. Skopiuj identyfikator wybranego modelu (np. `meta-llama/llama-3.1-8b-instruct:free`) — wkleisz go do zmiennej `MODEL` w wygenerowanym kodzie

#### Prompt dla modelu

```text
Kontekst:
Pracujesz w Google Colab i chcesz wywołać darmowy model przez OpenRouter API.

Wejście:
Brak — konfiguracja od zera.

Zadanie:
Zainstaluj pakiet `openai`. Następnie poproś użytkownika o wpisanie klucza OpenRouter przez `getpass.getpass` i utwórz klienta z adresem OpenRouter:

  client = openai.OpenAI(
      base_url="https://openrouter.ai/api/v1",
      api_key=_key,
  )

Zdefiniuj zmienną MODEL z nazwą modelu — tak, żeby łatwo ją zmienić w jednym miejscu:

  MODEL = "meta-llama/llama-3.1-8b-instruct:free"  # ← zmień tu, żeby użyć innego modelu

Na końcu wyślij jedno testowe żądanie używając MODEL i wydrukuj odpowiedź, żeby potwierdzić że połączenie działa.

Pokaż wynik:
- komunikat, że instalacja się powiodła,
- pole do wpisania klucza (bez jego wyświetlania),
- wypisany MODEL który będzie używany,
- krótka odpowiedź modelu na testowe zapytanie.

Warunek poprawności:
Model musi odpowiedzieć — nawet jednym zdaniem. Błąd 401 = problem z kluczem.

Jeśli wystąpi błąd:
Wyświetl treść błędu i podpowiedź: sprawdź, czy klucz jest aktywny na openrouter.ai.

Nie rób jeszcze:
Nie oceniaj kwestii dialogowych.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Potwierdzenie instalacji i pole do wpisania klucza API.
- Krótką odpowiedź modelu Llama potwierdzającą działające połączenie.

> **Jeśli chcesz użyć innego modelu darmowego:** zajrzyj na openrouter.ai/models, odfiltruj po „Free" i zamień nazwę modelu w Kroku 3A.

---
## Etap 3/6 — Ocena emocji dialogów

Model językowy przeanalizuje każdą kwestię i przypisze jej:
- **walencję** — liczbę od −1 (bardzo negatywna) do +1 (bardzo pozytywna),
- **emocję dyskretną** — jedną z sześciu: `radość`, `smutek`, `złość`, `strach`, `zaskoczenie`, `neutralna`.

Wysyłamy kwestie **partiami** — żeby nie przekroczyć limitu żądań per minutę.

In [ ]:
# === ETAP 3/6 · OCENA EMOCJI ===
assert 'dialogi' in dir(), "⛔ Najpierw uruchom Etap 1 — brak tabeli `dialogi`."
print(f"📍 Etap 3/6 · Ocena emocji. Wejście: {len(dialogi):,} kwestii do oceny.")

### Krok 3A. Batchowa ocena emocji

#### Cel i sens analityczny

Ocena linia po linii (jedno żądanie = jedna kwestia) jest wolna i kosztowna. Batching — przesyłanie wielu kwestii w jednym żądaniu i odebranie odpowiedzi jako tablicy JSON — jest znacznie wydajniejszy.

#### Twoja kolej: uzupełnij i uruchom

> **Instrukcja dla modelu (środkowy blok między `---`) jest wbudowana na stałe — nie zmieniaj jej. Wypełnij dwie oznaczone luki `[UZUPEŁNIJ: …]`, potem uruchom.**

In [ ]:
# TWOJA KOLEJ: wypełnij każdą lukę, potem uruchom tę komórkę.
moj_prompt = """
Kontekst:
Masz tabelę kwestii dialogowych w zmiennej `dialogi`. Masz skonfigurowanego klienta OpenRouter API (zmienna `client` — obiekt `openai.OpenAI` z `base_url="https://openrouter.ai/api/v1"`) oraz zmienną `MODEL` z nazwą wybranego modelu.

Wejście:
Tabela `dialogi` z kolumnami: numer sceny, postać, treść kwestii.

Zadanie:
Napisz funkcję przetwarzającą kwestie partiami po [UZUPEŁNIJ: ile kwestii na partię? — typowe wartości to 20–50]. Dla każdej partii wyślij do OpenRouter jedno żądanie (`client.chat.completions.create`, model=MODEL) z poniższą instrukcją wbudowaną na stałe w kod:

---
Oceń emocje poniższych kwestii dialogowych ze scenariusza filmowego.
Dla każdej kwestii zwróć obiekt JSON z polami:
- "id": numer porządkowy kwestii w tej partii (0-indeksowany),
- "walencja": liczba od -1.0 (bardzo negatywna) do 1.0 (bardzo pozytywna),
- "emocja": jedna z: radość, smutek, złość, strach, zaskoczenie, neutralna.
Zwróć WYŁĄCZNIE tablicę JSON, bez żadnego dodatkowego tekstu ani formatowania markdown.
Kwestie do oceny:
[TUTAJ WSTAW KWESTIE W FORMACIE: "id | postać: treść kwestii"]
---

Po odebraniu odpowiedzi sparsuj JSON i dołącz wyniki do tabeli kwestii jako nowe kolumny `walencja` i `emocja`. Zachowaj wynik jako zmienną o nazwie `emocje`. Jeśli parsowanie partii się nie powiedzie, wypełnij brakujące wiersze: walencja=0, emocja="neutralna" i wypisz ostrzeżenie. Dodaj opóźnienie [UZUPEŁNIJ: ile sekund między partiami? — 1–2 sekundy wystarczają] między partiami.

Pokaż wynik:
- komunikaty o przetwarzaniu kolejnych partii (numer partii i postęp),
- po zakończeniu: łączną liczbę ocenionych kwestii,
- próbkę 10 wierszy: postać | treść (pierwsze 60 znaków) | walencja | emocja.

Warunek poprawności:
Liczba ocenionych kwestii = liczba wierszy w `dialogi`. Kolumna `emocja` zawiera wyłącznie wartości z listy: radość, smutek, złość, strach, zaskoczenie, neutralna.

Jeśli wystąpi błąd:
Wypisz, przy której partii wystąpił problem i pokaż surową odpowiedź modelu.

Nie rób jeszcze:
Nie rysuj wykresów.
"""
assert "[UZUPEŁNIJ" not in moj_prompt, \
    "⛔ Zostały nieuzupełnione luki [UZUPEŁNIJ… — wypełnij każdą."
assert len(moj_prompt.strip()) > 200, \
    "⛔ Prompt za krótki — wypełnij sekcje konkretną treścią o SWOIM filmie."
assert "emocje" in moj_prompt, \
    "⛔ Upewnij się, że Twój prompt mówi modelowi o zapisaniu wyniku w zmiennej `emocje`."
print("✅ Prompt gotowy. Skopiuj poniższą treść do asystenta AI:\n")
print(moj_prompt)

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Komunikaty o przetwarzaniu kolejnych partii.
- Próbkę kwestii z przypisanymi walencjami i emocjami.

> **Wskazówka:** jeśli model zwraca JSON opakowany w znaczniki ` ```json ... ``` `, poproś go o poprawkę: *„usuń markdown z odpowiedzi — zwracaj tylko czystą tablicę JSON"*.

### Krok 3B. Podgląd i kontrola jakości

#### Cel i sens analityczny

Zanim narysujemy wykresy, sprawdzamy, czy rozkład emocji wygląda sensownie — czy model nie wpadł w schemat (np. wszystko „neutralna").

#### Twoja kolej: uzupełnij i uruchom

Przeczytaj sekcję **Cel i sens** powyżej i **Po uruchomieniu** poniżej. Wypełnij luki `[UZUPEŁNIJ: …]` w komórce kodu poniżej i uruchom ją — komórka sprawdzi i wypisze gotowy prompt.

In [ ]:
# TWOJA KOLEJ: wypełnij każdą lukę, potem uruchom tę komórkę.
moj_prompt = """
Kontekst:
Masz tabelę kwestii z przypisanymi emocjami i walencjami.

Wejście:
[UZUPEŁNIJ: jaka zmienna? jakie kolumny Cię interesują?]

Zadanie:
[UZUPEŁNIJ: co chcesz sprawdzić? — opisz, jakie statystyki i przykłady chcesz zobaczyć]

Pokaż wynik:
[UZUPEŁNIJ: w jakiej formie chcesz zobaczyć wyniki kontroli jakości?]

Warunek poprawności:
[UZUPEŁNIJ: jak rozpoznasz, że rozkład emocji jest zdegenerowany? — np. kiedy jedna emocja dominuje za bardzo?]

Jeśli wystąpi błąd:
Wyjaśnij, czy kolumna emocja jest pusta albo zawiera nieoczekiwane wartości.

Nie rób jeszcze:
Nie rysuj wykresów.
"""
assert "[UZUPEŁNIJ" not in moj_prompt, \
    "⛔ Zostały nieuzupełnione luki [UZUPEŁNIJ… — wypełnij każdą."
assert len(moj_prompt.strip()) > 200, \
    "⛔ Prompt za krótki — wypełnij sekcje konkretną treścią o SWOIM filmie."
print("✅ Prompt gotowy. Skopiuj poniższą treść do asystenta AI:\n")
print(moj_prompt)

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Tabelę: emocja | liczba kwestii | procent.
- Podstawowe statystyki walencji (min, mediana, max).
- 3 kwestie z najwyższą walencją i 3 z najniższą.

> **Jeśli rozkład jest zdegenerowany** (np. 90% „neutralna") — wróć do Kroku 3A i poproś model AI o zmodyfikowanie wbudowanej instrukcji oceny emocji, żeby była bardziej rozróżniająca.

---
## Etap 4/6 — Łuk emocjonalny filmu

Teraz patrzymy na emocje w czasie — jak zmienia się nastrój od pierwszej do ostatniej sceny. To tzw. *emotional arc*, znany z badań Reagana i in. (2016) oraz Vonneguta.

In [ ]:
# === ETAP 4/6 · ŁUK EMOCJONALNY ===
assert 'emocje' in dir(), "⛔ Najpierw uruchom Etap 3 — brak tabeli `emocje`."
print(f"📍 Etap 4/6 · Łuk emocjonalny. Wejście: `emocje` ({len(emocje):,} kwestii z walencją).")

In [ ]:
# HIPOTEZA — wpisz ZANIM narysujesz wykres
# Gdzie w Twoim filmie będzie najgłębsza dolina emocjonalna (najbardziej mroczna sekwencja)?
# Opisz scenę lub moment fabuły.
moja_hipoteza_4 = ""   # np. "środkowa część — scena konfrontacji głównych bohaterów"
assert moja_hipoteza_4.strip(), \
    "⛔ Najpierw opisz, gdzie spodziewasz się emocjonalnego dołka — zanim zobaczysz wykres."
print("✏️  Hipoteza zapisana. Teraz wygeneruj wykres łuku emocjonalnego.")

### Krok 4A. Wykres łuku emocjonalnego

#### Cel i sens analityczny

Zamiast patrzeć na pojedyncze kwestie, uśredniamy walencję w oknie przesuwnym scen. Wygładzony wykres pokazuje, gdzie film jest napięty i mroczny, a gdzie emocje się rozładowują.

#### Twoja kolej: uzupełnij i uruchom

Przeczytaj **Cel i sens** (powyżej) i **Po uruchomieniu** (poniżej). Wypełnij luki `[UZUPEŁNIJ: …]` w komórce kodu poniżej i uruchom.

In [ ]:
# TWOJA KOLEJ: wypełnij każdą lukę, potem uruchom tę komórkę.
moj_prompt = """
Kontekst:
Masz tabelę kwestii z walencją i numerem sceny — chcesz zobaczyć, jak zmienia się nastrój scenariusza w czasie.

Wejście:
[UZUPEŁNIJ: jaka zmienna? jakie dwie kolumny są potrzebne?]

Zadanie:
[UZUPEŁNIJ: opisz krok po kroku — (1) jak wyliczyć walencję na scenę, (2) jak wygładzić (okno o jakiej szerokości?), (3) co narysować, co ma być na osiach, jak zaznaczyć linię zerową]

Pokaż wynik:
[UZUPEŁNIJ: jakie elementy musi zawierać wykres? co powinno być widoczne na osiach?]

Warunek poprawności:
Wykres nie powinien być płaski — wypisz ostrzeżenie, jeśli walencja przez cały film jest bliska 0.

Jeśli wystąpi błąd:
Wyjaśnij, czy tabela nie ma kolumny walencja albo czy wszystkie numery scen są identyczne.

Nie rób jeszcze:
Nie rysuj wykresu emocja × płeć.
"""
assert "[UZUPEŁNIJ" not in moj_prompt, \
    "⛔ Zostały nieuzupełnione luki [UZUPEŁNIJ… — wypełnij każdą."
assert len(moj_prompt.strip()) > 200, \
    "⛔ Prompt za krótki — wypełnij sekcje konkretną treścią o SWOIM filmie."
print("✅ Prompt gotowy. Skopiuj poniższą treść do asystenta AI:\n")
print(moj_prompt)

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Wykres liniowy z trajektorią emocjonalną od pierwszej do ostatniej sceny.
- Wyraźne wzloty i upadki nastroju.
- Poziomą linię zerową oddzielającą nastrój pozytywny od negatywnego.

In [ ]:
# CHECKPOINT — uzupełnij na podstawie wykresu powyżej
numer_doliny = ""  # np. "45" — numer sceny z najgłębszą doliną (odczytaj z wykresu)
assert numer_doliny.strip().isdigit(), \
    "⛔ Wpisz numer sceny z najniższą walencją (odczytaj z wykresu)."
print(f"📊 Checkpoint: najgłębsza dolina emocjonalna — scena {numer_doliny}.")
print(f"Twoja hipoteza mówiła o: '{moja_hipoteza_4[:80]}'")
print("Porównaj z hipotezą powyżej.")

#### Pytanie interpretacyjne

Gdzie na wykresie są doliny i szczyty? Czy pokrywają się z ważnymi zwrotami fabuły, które pamiętasz z filmu? Vonnegut opisywał opowieści jako „kształty" — jaki kształt ma Twój film?

---
## Etap 5/6 — Emocje według płci

Łączymy dwa wymiary: emocje i płeć. Pytamy, czy kobiety i mężczyźni w tym scenariuszu wyrażają inne emocje.

In [ ]:
# === ETAP 5/6 · EMOCJE × PŁEĆ ===
assert 'emocje' in dir(), "⛔ Najpierw uruchom Etap 3 — brak tabeli `emocje`."
assert 'postacie' in dir(), "⛔ Najpierw uruchom Etap 1 — brak tabeli `postacie`."
print(f"📍 Etap 5/6 · Emocje × płeć. "
      f"Wejście: {len(emocje):,} kwestii i {len(postacie)} postaci z płcią.")

### Krok 5A. Wykres rozkładu emocji według płci

#### Cel i sens analityczny

Kino przez dekady utrwalało stereotypy emocjonalne: kobiety płaczą, mężczyźni się złoszczą. Analiza danych pozwala sprawdzić, czy konkretny scenariusz reprodukuje te wzorce. Pamiętaj o granicy: to, co postać *mówi*, nie musi odzwierciedlać jej głębszych emocji.

#### Twoja kolej: uzupełnij i uruchom

Przeczytaj **Cel i sens** (powyżej) i **Po uruchomieniu** (poniżej). Wypełnij luki `[UZUPEŁNIJ: …]` w komórce kodu poniżej i uruchom.

In [ ]:
# TWOJA KOLEJ: wypełnij każdą lukę, potem uruchom tę komórkę.
moj_prompt = """
Kontekst:
Masz tabelę kwestii z kolumną emocja i kolumną postaci. Masz tabelę postaci z kolumną płeć (K/M/?).

Wejście:
[UZUPEŁNIJ: jakie dwie zmienne łączysz? po jakiej kolumnie je łączysz?]

Zadanie:
[UZUPEŁNIJ: (1) jak połączyć tabele, żeby każda kwestia miała płeć postaci, (2) co policzyć (pamiętaj o normalizacji do %), (3) jakie postacie pominąć, (4) co narysować]

Pokaż wynik:
[UZUPEŁNIJ: jaki typ wykresu? co na osiach? jak pokazać obie płcie jednocześnie?]

Warunek poprawności:
Wartości procentowe dla każdej płci sumują się do 100%. Jeśli jedna płeć ma mniej niż 10 kwestii, wypisz ostrzeżenie.

Jeśli wystąpi błąd:
Wyjaśnij, czy problem dotyczy łączenia tabel albo brakujących wartości płci.

Nie rób jeszcze:
Nie zapisuj pliku.
"""
assert "[UZUPEŁNIJ" not in moj_prompt, \
    "⛔ Zostały nieuzupełnione luki [UZUPEŁNIJ… — wypełnij każdą."
assert len(moj_prompt.strip()) > 200, \
    "⛔ Prompt za krótki — wypełnij sekcje konkretną treścią o SWOIM filmie."
print("✅ Prompt gotowy. Skopiuj poniższą treść do asystenta AI:\n")
print(moj_prompt)

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Wykres porównujący rozkład emocji między kobietami i mężczyznami.
- Wartości procentowe (umożliwiające porównanie mimo różnej liczby kwestii).
- Wyraźne etykiety płci i kategorii emocji.

#### Pytanie interpretacyjne

Czy rozkład emocji różni się między płciami? Które emocje są „sfeminizowane" lub „zmaskulinizowane" w tym konkretnym filmie? Jak ten wynik ma się do Twoich intuicji z oglądania?

> **Zastrzeżenie metodologiczne:** emocja przypisana przez model to interpretacja tekstu dialogu — nie subiektywne odczucie postaci ani intencja autorska. Wynik mówi coś o *reprezentacji*, nie o *rzeczywistości*.

---
## Etap 6/6 — Eksport danych

Zapisujemy wyniki oceny emocji do pliku wejściowego dla NB3.

In [ ]:
# === ETAP 6/6 · EKSPORT ===
assert 'emocje' in dir(), "⛔ Najpierw uruchom Etap 3 — brak tabeli `emocje`."
print(f"📍 Etap 6/6 · Eksport. `emocje` ({len(emocje):,} wierszy) → pliki CSV.")

### Krok 6A. Zapis pliku `emocje.csv`

#### Cel i sens analityczny

`emocje.csv` to „wzbogacona" wersja `dialogi.csv` — zawiera kwestie z ocenami emocji. `emocje_postaci.csv` to emocjonalne portrety postaci. W NB3 dołączymy te dane do sieci postaci.

#### Prompt dla modelu

```text
Kontekst:
Masz tabelę kwestii z ocenionymi emocjami i walencjami w zmiennej `emocje`.

Wejście:
Tabela `emocje` z kolumnami: numer sceny, postać, treść kwestii, liczba słów, walencja, emocja.

Zadanie:
Zapisz tę tabelę do pliku `emocje.csv` w kodowaniu UTF-8. Następnie wygeneruj skrócony plik `emocje_postaci.csv`, w którym każda postać ma jeden wiersz z kolumnami: postać, dominująca emocja (najczęstsza), średnia walencja, liczba kwestii. Pokaż jak pobrać oba pliki z Colab.

Pokaż wynik:
- komunikat o zapisaniu obu plików z liczbą wierszy,
- 5 pierwszych wierszy `emocje_postaci.csv`,
- wskazówkę jak pobrać pliki.

Warunek poprawności:
Oba pliki niepuste. Liczba wierszy w `emocje.csv` = liczba kwestii.

Jeśli wystąpi błąd:
Wyjaśnij, którego pliku nie udało się zapisać i dlaczego.

Nie rób już nic więcej.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Potwierdzenie zapisania `emocje.csv` i `emocje_postaci.csv`.
- Podgląd emocjonalnych portretów postaci.
- Wskazówkę jak pobrać pliki z Colab.

---
## Co dalej?

- Masz teraz `emocje.csv` (kwestie z walencją i emocją) i `emocje_postaci.csv` (emocjonalne portrety postaci).
- W **NB3** połączysz te dane z siecią z modułu 1 i wygenerujesz prezentację HTML z czterema wykresami.
- Jeśli łuk emocjonalny jest zbyt płaski albo rozkład emocji zdegenerowany, wróć do Kroku 3A i zmodyfikuj wbudowaną instrukcję oceny emocji — iteracja jest częścią metody.